In [1]:
import re
import unicodedata
import pandas as pd
# New

In [2]:
FANTASY_CSV = "fantasy_optimizer.csv"
STATS_CSV = "players_data-2025_2026.csv"
OUT_CSV = "fantasy_enriched.csv"

MAX_EDIT_DIST = 1

In [3]:
fantasy = pd.read_csv(FANTASY_CSV)
print(f"Total players: {len(fantasy)}")
print(fantasy["status"].value_counts().to_string())

fantasy = fantasy[fantasy["status"] != "transferred"].reset_index(drop=True)
print(f"\nAfter removing transferred: {len(fantasy)}")
print(fantasy.groupby("team")["name"].count().sort_values(ascending=False).to_string())

stats = pd.read_csv(STATS_CSV)

Total players: 654
status
playing        518
transferred    134
injured          1
suspended        1

After removing transferred: 520
team
Algeria        26
Argentina      26
Australia      26
Austria        26
Brazil         26
Cabo Verde     26
Canada         26
Colombia       26
Croatia        26
Egypt          26
England        26
France         26
Ghana          26
Mexico         26
Morocco        26
Norway         26
Paraguay       26
Portugal       26
Spain          26
Switzerland    26


In [4]:
def norm(s):
    if pd.isna(s): # normalize text into lower case no accents
        return ""
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z ]", "", re.sub(r"\s+", " ", s)).strip()

def lev(a, b):    # computing Levenshtein distance
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for ca in a:
        curr = [prev[0] + 1]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[-1] + 1, prev[j-1] + (ca != cb)))
        prev = curr
    return prev[-1]

def best_match(name, pool):    # finds closest match in pool for name using Levenshtein distance
    fn = norm(name)
    best_dist, best_idx = MAX_EDIT_DIST + 1, None
    for idx, cn in zip(pool.index, pool["_name_norm"]):
        if abs(len(cn) - len(fn)) > MAX_EDIT_DIST:
            continue
        d = lev(fn, cn)
        if d < best_dist:
            best_dist, best_idx = d, idx
    if best_idx is not None and best_dist <= MAX_EDIT_DIST:
        return best_idx, best_dist
    return None, None


In [5]:

stats["fbref_code"] = stats["Nation"].str.extract(r"([A-Z]{2,4})$") #  new column fbref_code

COUNT_COLS = [c for c in [
    "MP", "Starts", "Min", "90s",
    "Gls", "Ast", "G+A", "xG", "xAG", "npxG", "G-PK",
    "Tkl", "TklW", "Blocks", "Int", "Tkl+Int", "Clr", "Err",
    "PrgP", "PrgC", "KP", "PPA", "xA", "Ast_stats_passing",
    "GA", "Saves", "CS", "PKA", "PKsv",
    "Touches", "Carries", "PrgR", "Mis", "Dis",
    "CrdY", "CrdR", "PKwon", "PKcon", "Recov",
    "PK", "PKatt", "SoTA", "PKm", "Sh", "SoT",
    "Fls", "Fld", "Off", "Crs", "OG", "2CrdY",
] if c in stats.columns]

RATE_COLS = [c for c in [
    "Cmp%_stats_passing", "Save%", "CS%",
    "GA90", "SoT%", "Sh/90", "SoT/90", "G/Sh", "G/SoT",
] if c in stats.columns]

META_COLS = [c for c in ["Pos", "Squad", "Comp", "Age"] if c in stats.columns]

agg_dict = {c: "sum" for c in COUNT_COLS} # sums integer columns
agg_dict.update({c: "first" for c in META_COLS}) # takes first value for meta columns

if "Squad" in META_COLS:
    agg_dict["Squad"] = lambda x: " / ".join(x.dropna().astype(str).unique())


stats_agg = stats.groupby(["Player", "fbref_code"], as_index=False, sort=False).agg(agg_dict)

rates = stats.loc[
    stats.groupby(["Player", "fbref_code"])["MP"].idxmax(),
    ["Player", "fbref_code"] + RATE_COLS   # preserves rates of club where player played most
]

stats_agg = stats_agg.merge(rates, on=["Player", "fbref_code"], how="left")

stats_agg["fbref_multi_club"] = stats.groupby(["Player", "fbref_code"]).size().gt(1).values # players with multiple clubs in the same season

stats_agg["_name_norm"] = stats_agg["Player"].apply(norm) #


print(f"Stats aggregated: {len(stats_agg)} players")

Stats aggregated: 2685 players


In [6]:
NATION_MAP = {
    "Algeria": "ALG", "Argentina": "ARG", "Australia": "AUS", "Austria": "AUT",
    "Belgium": "BEL", "Bosnia and Herzegovina": "BIH", "Brazil": "BRA",
    "Cabo Verde": "CPV", "Canada": "CAN", "Colombia": "COL", "Congo DR": "COD",
    "Croatia": "CRO", "Curacao": "CUW", "Czechia": "CZE", "Ecuador": "ECU",
    "Egypt": "EGY", "England": "ENG", "France": "FRA", "Germany": "GER",
    "Ghana": "GHA", "Haiti": "HAI", "IR Iran": "IRN", "Iraq": "IRQ",
    "Japan": "JPN", "Jordan": "JOR", "Korea Republic": "KOR", "Mexico": "MEX",
    "Morocco": "MAR", "Netherlands": "NED", "New Zealand": "NZL", "Norway": "NOR",
    "Panama": "PAN", "Paraguay": "PAR", "Portugal": "POR", "Qatar": "QAT",
    "Saudi Arabia": "KSA", "Scotland": "SCO", "Senegal": "SEN",
    "South Africa": "RSA", "Spain": "ESP", "Sweden": "SWE", "Switzerland": "SUI",
    "Tunisia": "TUN", "Turkiye": "TUR", "Uruguay": "URU", "USA": "USA",
    "Uzbekistan": "UZB", "Cote d Ivoire": "CIV",
}

fantasy["fbref_code"] = fantasy["team"].map(NATION_MAP)

unmapped = fantasy.loc[fantasy["fbref_code"].isna(), "team"].dropna().unique()
if len(unmapped):
    print(f"\nUnmapped teams: {list(unmapped)}")
else:
    print("\n No team unmapped")


 No team unmapped


In [7]:
codes = set(fantasy["fbref_code"].dropna().unique())

stats_filtered = stats_agg[stats_agg["fbref_code"].isin(codes)].copy()

print(f"Stats rows for tournament nations: {len(stats_filtered)}")   # keep only players stats whose nation in the world cup.

Stats rows for tournament nations: 1391


In [8]:
ADD_COLS = COUNT_COLS + RATE_COLS + META_COLS + ["fbref_multi_club"]
for col in ADD_COLS:
    fantasy[f"club_{col}"] = pd.NA
fantasy["clubstats_matched_player"] = pd.NA
fantasy["clubs_match_dist"] = pd.NA

match_log = []  # match log

for code, grp in fantasy.groupby("fbref_code", dropna=True):
    pool = stats_filtered[stats_filtered["fbref_code"] == code]  # only matches inside nation
    if pool.empty:
        continue
    for idx in grp.index:
        fname = fantasy.at[idx, "name"]
        match_idx, dist = best_match(fname, pool)
        if match_idx is None:
            continue
        row = stats_filtered.loc[match_idx]
        for col in ADD_COLS:
            if col in row.index:
                fantasy.at[idx, f"club_{col}"] = row[col]
        fantasy.at[idx, "clubstats_matched_player"] = row["Player"]
        fantasy.at[idx, "clubs_match_dist"] = dist
        match_log.append({"nation": code, "fantasy": fname, "club": row["Player"], "dist": dist})

In [9]:
matched = fantasy["clubs_match_dist"].notna().sum()
print(f"\nMatched {matched} out of {len(fantasy)} ({matched/len(fantasy):.1%})")

match_df = pd.DataFrame(match_log)
if not match_df.empty:
    for d in range(MAX_EDIT_DIST + 1):
        print(f"  dist={d}: {(match_df['dist']==d).sum()}")

    suspect = match_df[match_df["dist"] > 0].sort_values(["dist","nation"])
    if not suspect.empty:
        print(f"\nImperfect matches (dist > 0)")
        print(suspect[["nation","fantasy","club","dist"]].to_string(index=False))

print(f"\nMatch rate by nation:")
for code, grp in fantasy.groupby("fbref_code", dropna=True):
    n_matched = grp["clubs_match_dist"].notna().sum()
    n_total = len(grp)
    flag = " no top-5 data" if n_matched == 0 else ""
    print(f"  {code}: {n_matched} in {n_total} ({n_matched/n_total:.0%}){flag}")

unmatched = fantasy[fantasy["clubs_match_dist"].isna()][["name","team","position"]]


Matched 266 out of 520 (51.2%)
  dist=0: 261
  dist=1: 5

Imperfect matches (dist > 0)
nation                     fantasy                        club  dist
   ALG             Mohammed Amoura              Mohamed Amoura     1
   BRA               Luiz Henrique               Luis Henrique     1
   CRO               Marco Pasalic               Mario Pašalić     1
   ESP                 Yéremy Pino                 Yeremi Pino     1
   MAR Ayoube Amaimouni-Echghouyab Ayoube Amaimouni Echghouyab     1

Match rate by nation:
  ALG: 11 in 26 (42%)
  ARG: 18 in 26 (69%)
  AUS: 4 in 26 (15%)
  AUT: 17 in 26 (65%)
  BRA: 15 in 26 (58%)
  CAN: 7 in 26 (27%)
  COL: 8 in 26 (31%)
  CPV: 1 in 26 (4%)
  CRO: 18 in 26 (69%)
  EGY: 3 in 26 (12%)
  ENG: 25 in 26 (96%)
  ESP: 24 in 26 (92%)
  FRA: 24 in 26 (92%)
  GHA: 13 in 26 (50%)
  MAR: 13 in 26 (50%)
  MEX: 5 in 26 (19%)
  NOR: 16 in 26 (62%)
  PAR: 4 in 26 (15%)
  POR: 17 in 26 (65%)
  SUI: 23 in 26 (88%)


In [10]:
fantasy = fantasy.drop(columns=["fbref_code"], errors="ignore")

fantasy.to_csv(OUT_CSV, index=False)

print(f"\nSaved {OUT_CSV} {fantasy.shape[0]} rows x {fantasy.shape[1]} cols")



Saved fantasy_enriched.csv 520 rows x 106 cols


| Column | Type | Description |
|---|---|---|
| `fifa_id` | int | FIFA fantasy platform unique player identifier |
| `name` | str | Player display name from FIFA fantasy (knownName or firstName + lastName) |
| `squad_id` | int | FIFA fantasy internal squad/nation identifier |
| `team` | str | Nation name in English, mapped from squad_id |
| `position` | str | Fantasy position: DEF, MID, FWD, GK |
| `price` | float | Fantasy selection price in FIFA fantasy currency |
| `status` | str | Player availability: playing, transferred |
| `total_points` | int | Total fantasy points accumulated across all completed rounds |
| `avg_points` | float | Average fantasy points per round played |
| `matches_played` | int | Number of rounds with non-empty stat entries |
| `percent_selected` | float | Percentage of fantasy teams that have selected this player |
| `round_1_points` | int | Fantasy points earned in round 1 |
| `round_2_points` | int | Fantasy points earned in round 2 |
| `round_3_points` | int | Fantasy points earned in round 3 |
| `round_4_points` | int | Fantasy points earned in round 4 |
| `next_fixture` | int | FIFA fantasy fixture ID for the next scheduled match |

| Column | Type | Description |
|---|---|---|
| `betano_matched_name` | str | Player name as it appears in Betano scorer markets |
| `betano_match_score` | float | Fuzzy match confidence score 0 to 100; 100 indicates exact or manual override match |

| Column | Type | Description |
|---|---|---|
| `anytime_scorer_odd` | float | Betano decimal odd for player to score at any point; defaults to 150.0 if team has a match but player has no Betano entry |
| `anytime_scorer_prob` | float | Raw implied probability from anytime scorer odd (1 / odd) |
| `first_scorer_odd` | float | Betano decimal odd for player to score the first goal of the match |
| `first_scorer_prob` | float | Raw implied probability from first scorer odd (1 / odd) |
| `last_scorer_odd` | float | Betano decimal odd for player to score the last goal of the match |

| Column | Type | Description |
|---|---|---|
| `match_home` | str | Home team name for the player's next match |
| `match_away` | str | Away team name for the player's next match |
| `is_home` | bool | True if the player's team is the home side |
| `opponent` | str | Opposing team name |
| `match_home_win_odd` | float | Betano decimal odd for home team victory |
| `match_draw_odd` | float | Betano decimal odd for a draw |
| `match_away_win_odd` | float | Betano decimal odd for away team victory |
| `match_btts_prob` | float | De-vigged probability that both teams score |
| `match_over_25_odd` | float | Betano decimal odd for over 2.5 total goals |
| `match_over_25_prob` | float | De-vigged probability of over 2.5 total goals |
| `match_over_35_odd` | float | Betano decimal odd for over 3.5 total goals |
| `team_over_05_odd` | float | Betano decimal odd for the player's team to score at least 1 goal |
| `team_over_15_odd` | float | Betano decimal odd for the player's team to score at least 2 goals |
| `team_score_prob` | float | De-vigged probability the player's team scores at least once |
| `team_score_2_prob` | float | Implied probability the player's team scores 2 or more goals (1 / team_over_15_odd) |
| `team_cs_prob` | float | De-vigged probability the player's team keeps a clean sheet |
| `team_qualify_odd` | float | Betano decimal odd for the player's team to advance to the next round |
| `team_win2_odd` | float | Betano decimal odd for the player's team to win by 2 or more goals |
| `opp_over_05_odd` | float | Betano decimal odd for the opponent to score at least 1 goal |
| `opp_score_prob` | float | De-vigged probability the opponent scores at least once |
| `opp_cs_prob` | float | De-vigged probability the opponent keeps a clean sheet |

| Column | Type | Description |
|---|---|---|
| `stat_GS` | int | Goals scored during the tournament (FIFA fantasy stat) |
| `stat_AS` | int | Assists during the tournament (FIFA fantasy stat) |
| `stat_CS` | int | Clean sheets during the tournament (FIFA fantasy stat) |
| `stat_GC` | int | Goals conceded during the tournament (FIFA fantasy stat) |
| `stat_MP` | int | Minutes played during the tournament (FIFA fantasy stat) |
| `stat_YC` | int | Yellow cards during the tournament (FIFA fantasy stat) |
| `stat_RC` | int | Red cards during the tournament (FIFA fantasy stat) |
| `stat_ST` | int | Shots on target during the tournament (FIFA fantasy stat) |
| `stat_SB` | int | Shots blocked during the tournament (FIFA fantasy stat) |
| `stat_CC` | int | Chances created during the tournament (FIFA fantasy stat) |
| `stat_PS` | int | Penalties saved during the tournament (FIFA fantasy stat, GK) |
| `stat_T` | int | Tackles during the tournament (FIFA fantasy stat) |
| `stat_S` | int | Saves during the tournament (FIFA fantasy stat, GK) |
| `stat_SXI` | int | Number of times named in the starting XI (FIFA fantasy stat) |
| `stat_OG` | int | Own goals during the tournament (FIFA fantasy stat) |
| `stat_PC` | int | Penalties committed during the tournament (FIFA fantasy stat) |
| `stat_PW` | int | Penalties won during the tournament (FIFA fantasy stat) |
| `stat_FK` | int | Free kicks earned during the tournament (FIFA fantasy stat) |


| Column                  | Type  | Description                                                         |
| ----------------------- | ----- | ------------------------------------------------------------------- |
| `club_MP`               | int   | Matches played in top 5 European leagues, 2025/26 season            |
| `club_Starts`           | int   | Matches started in top 5 European leagues                           |
| `club_Min`              | int   | Total minutes played in top 5 European leagues                      |
| `club_90s`              | float | Minutes played expressed as full 90-minute equivalents              |
| `club_Gls`              | int   | Goals scored in top 5 European leagues                              |
| `club_Ast`              | int   | Assists in top 5 European leagues                                   |
| `club_G+A`              | int   | Goals plus assists                                                  |
| `club_G-PK`             | int   | Non-penalty goals                                                   |
| `club_TklW`             | int   | Tackles won                                                         |
| `club_Int`              | int   | Interceptions                                                       |
| `club_GA`               | int   | Goals conceded (GK only)                                            |
| `club_Saves`            | int   | Saves (GK only)                                                     |
| `club_CS`               | int   | Clean sheets (GK only)                                              |
| `club_PKA`              | int   | Penalties faced (GK only)                                           |
| `club_PKsv`             | int   | Penalties saved (GK only)                                           |
| `club_CrdY`             | int   | Yellow cards                                                        |
| `club_CrdR`             | int   | Red cards                                                           |
| `club_PK`               | int   | Penalties scored                                                    |
| `club_PKatt`            | int   | Penalty attempts                                                    |
| `club_SoTA`             | int   | Shots on target faced (GK only)                                     |
| `club_PKm`              | int   | Penalties missed                                                    |
| `club_Sh`               | int   | Shots                                                               |
| `club_SoT`              | int   | Shots on target                                                     |
| `club_Fls`              | int   | Fouls committed                                                     |
| `club_Fld`              | int   | Fouls drawn                                                         |
| `club_Off`              | int   | Offsides                                                            |
| `club_Crs`              | int   | Crosses                                                             |
| `club_OG`               | int   | Own goals                                                           |
| `club_2CrdY`            | int   | Second yellow cards                                                 |
| `club_Save%`            | float | Save percentage (GK); unreliable if multi-club                      |
| `club_CS%`              | float | Clean sheet percentage; unreliable if multi-club                    |
| `club_GA90`             | float | Goals conceded per 90 (GK)                                          |
| `club_SoT%`             | float | Shots on target %                                                   |
| `club_Sh/90`            | float | Shots per 90                                                        |
| `club_SoT/90`           | float | Shots on target per 90                                              |
| `club_G/Sh`             | float | Goals per shot                                                      |
| `club_G/SoT`            | float | Goals per shot on target                                            |
| `club_Pos`              | str   | Position code (FW, MF, DF, GK)                                      |
| `club_Squad`            | str   | Clubs played for (slash-separated if multiple)                      |
| `club_Comp`             | str   | Competitions played in                                              |
| `club_Age`              | int   | Player age                                                          |
| `club_fbref_multi_club` | bool  | True if multiple clubs in season; rate stats unreliable if True     |
| `club_matched_player`   | str   | FBref player name matched via fuzzy match                           |
| `club_match_dist`       | int   | Edit distance between names (0 = exact match, i.e. 100% confidence) |

